In [1]:
import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
job_path = "../data/raw/Job_listing_dataset/jobs_data (1).xlsx"

jobs = pd.read_excel(job_path)

print("Job dataset loaded successfully!")
print("Number of jobs:", len(jobs))
print("Number of columns:", len(jobs.columns))

jobs.head()

Job dataset loaded successfully!
Number of jobs: 61773
Number of columns: 12


,job_id,job_role,company,experience,salary,location,rating,reviews,key_skills,posted_on,job_link,company_link
0,181224917811,GN - Strategy - MC - SC&O - SC Digital Core Ar...,Accenture,3-6 Yrs,Not disclosed,"Hyderabad, Chennai, Bengaluru",3.9,52479 Reviews,"sap ariba,presentation skills,solution impleme...",1 Day Ago,https://www.naukri.com/job-listings-gn-strateg...,https://www.naukri.com/accenture-jobs-careers-...
1,181224906322,GN - SC&O - BPM - Senior Manager,Accenture,2-4 Yrs,Not disclosed,"Gurugram, Bengaluru, Delhi / NCR",3.9,52479 Reviews,"spend analysis,market research,sales and opera...",1 Day Ago,https://www.naukri.com/job-listings-gn-sc-o-bp...,https://www.naukri.com/accenture-jobs-careers-...
2,181224906320,GN - SC&O - S&P - CLM - Associate Manager,Accenture,1-5 Yrs,Not disclosed,"Gurugram, Bengaluru, Delhi / NCR",3.9,52479 Reviews,"Sourcing,PowerBI,procurement,digital sourcing,...",1 Day Ago,https://www.naukri.com/job-listings-gn-sc-o-s-...,https://www.naukri.com/accenture-jobs-careers-...
3,181224904080,GN - SONG - Service - Command Center of Future...,Accenture,12-15 Yrs,Not disclosed,"New Delhi, Gurugram, Bengaluru",3.9,52479 Reviews,"cloud solutions,WFM solutions,AI,Customer Enga...",1 Day Ago,https://www.naukri.com/job-listings-gn-song-se...,https://www.naukri.com/accenture-jobs-careers-...
4,181224904079,GN - SC&O - S&P - CLM - Manager,Accenture,8-13 Yrs,Not disclosed,"New Delhi, Gurugram, Bengaluru",3.9,52479 Reviews,"Procurement,Sourcing,Supply Chain Management,p...",1 Day Ago,https://www.naukri.com/job-listings-gn-sc-o-s-...,https://www.naukri.com/accenture-jobs-careers-...


In [3]:
jobs["job_text"] = (
    jobs["job_role"].fillna("") + " " +
    jobs["key_skills"].fillna("") + " " +
    jobs["experience"].fillna("") + " " +
    jobs["location"].fillna("")
)

print("Combined job text created successfully!")
jobs[["job_role", "key_skills", "job_text"]].head()

Combined job text created successfully!


,job_role,key_skills,job_text
0,GN - Strategy - MC - SC&O - SC Digital Core Ar...,"sap ariba,presentation skills,solution impleme...",GN - Strategy - MC - SC&O - SC Digital Core Ar...
1,GN - SC&O - BPM - Senior Manager,"spend analysis,market research,sales and opera...",GN - SC&O - BPM - Senior Manager spend analysi...
2,GN - SC&O - S&P - CLM - Associate Manager,"Sourcing,PowerBI,procurement,digital sourcing,...",GN - SC&O - S&P - CLM - Associate Manager Sour...
3,GN - SONG - Service - Command Center of Future...,"cloud solutions,WFM solutions,AI,Customer Enga...",GN - SONG - Service - Command Center of Future...
4,GN - SC&O - S&P - CLM - Manager,"Procurement,Sourcing,Supply Chain Management,p...","GN - SC&O - S&P - CLM - Manager Procurement,So..."


In [4]:
job_tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000
)

job_matrix = job_tfidf.fit_transform(jobs["job_text"])

print("Job TF-IDF created successfully!")
print("TF-IDF matrix shape:", job_matrix.shape)

Job TF-IDF created successfully!
TF-IDF matrix shape: (61773, 10000)


In [5]:
resume_df = pd.read_csv("../data/processed/cleaned_resumes.csv")

test_resume = resume_df.iloc[0]["Resume_Text"]

print("Test resume loaded successfully!")
print("Resume category:", resume_df.iloc[0]["Category"])
print("\nResume text preview:")
print(test_resume[:500])

Test resume loaded successfully!
Resume category: ACCOUNTANT

Resume text preview:
ACCOUNTANT
Summary
Financial Accountant specializing in financial planning, reporting and analysis within the Department of Defense.
Highlights
Account reconciliations
Results-oriented
Financial reporting
Critical thinking
Accounting operations professional
Analysis of financial systems
ERP (Enterprise Resource Planning) software.
Excellent facilitator
Accomplishments
Served on a tiger team which identified and resolved General Ledger postings in DEAMS totaling $360B in accounting adjustments. T


In [6]:
resume_vector = job_tfidf.transform([test_resume])

print("Resume converted to TF-IDF successfully!")
print("Resume vector shape:", resume_vector.shape)

Resume converted to TF-IDF successfully!
Resume vector shape: (1, 10000)


In [7]:
similarity_scores = cosine_similarity(resume_vector, job_matrix)

print("Similarity calculated successfully!")
print("Similarity matrix shape:", similarity_scores.shape)

Similarity calculated successfully!
Similarity matrix shape: (1, 61773)


In [8]:
top_10_indices = similarity_scores[0].argsort()[-10:][::-1]

top_jobs = jobs.iloc[top_10_indices].copy()

top_jobs["match_score"] = similarity_scores[0][top_10_indices] * 100

print("Top 10 recommended jobs:")
top_jobs[["job_role", "company", "location", "key_skills", "match_score"]]

Top 10 recommended jobs:


,job_role,company,location,key_skills,match_score
31786,Chief Financial Officer,Aaiji Group,Pune,"Accounts And Finance,Management Accounting,Fin...",26.870244
36937,Finance Control Manager,Geeta University Kr Education Society,Samalkha,"Accounts Reconciliation,Management Accounting,...",25.476675
57942,Accounts Associate,NaN,Kochi,"Accounting Operations,Financial Accounting,Acc...",25.139775
25688,Manager Finance and Accounts,Walking Tree (india),"Mumbai Suburban, Mumbai (All Areas)","Team Management,General Accounting,Accounts An...",24.914939
47521,Accountant Staff,NaN,Chennai(East Tambaram),"Corporate Accounting,Branch Accounting,Balance...",24.705736
59009,Account & Finance,NaN,Chandigarh,"Finance And Accounts,Financial Accounting,Fina...",24.699910
32361,Accountant,NaN,"Mumbai(Nariman Point), Mumbai Suburban","Accounting,Financial Reporting,Accounting Oper...",24.682140
21330,Manager - US Accounting,NaN,Mumbai,"US Accounting,financial operations,accounting ...",24.190819
33532,Accounting Senior Executive,Ushodaya Enterprises,Hyderabad,"SAP,Bank Reconciliation,Accounts Payable,Compl...",24.019906
37908,Accounts Executive,Laser Power And Infra,Kolkata,"Management accounting,Excel,Financial reportin...",22.872482


In [9]:
recommendations = top_jobs[
    ["job_role", "company", "location", "key_skills", "match_score"]
].reset_index(drop=True)

recommendations.index = recommendations.index + 1

recommendations

,job_role,company,location,key_skills,match_score
1,Chief Financial Officer,Aaiji Group,Pune,"Accounts And Finance,Management Accounting,Fin...",26.870244
2,Finance Control Manager,Geeta University Kr Education Society,Samalkha,"Accounts Reconciliation,Management Accounting,...",25.476675
3,Accounts Associate,NaN,Kochi,"Accounting Operations,Financial Accounting,Acc...",25.139775
4,Manager Finance and Accounts,Walking Tree (india),"Mumbai Suburban, Mumbai (All Areas)","Team Management,General Accounting,Accounts An...",24.914939
5,Accountant Staff,NaN,Chennai(East Tambaram),"Corporate Accounting,Branch Accounting,Balance...",24.705736
6,Account & Finance,NaN,Chandigarh,"Finance And Accounts,Financial Accounting,Fina...",24.699910
7,Accountant,NaN,"Mumbai(Nariman Point), Mumbai Suburban","Accounting,Financial Reporting,Accounting Oper...",24.682140
8,Manager - US Accounting,NaN,Mumbai,"US Accounting,financial operations,accounting ...",24.190819
9,Accounting Senior Executive,Ushodaya Enterprises,Hyderabad,"SAP,Bank Reconciliation,Accounts Payable,Compl...",24.019906
10,Accounts Executive,Laser Power And Infra,Kolkata,"Management accounting,Excel,Financial reportin...",22.872482


In [12]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9+#]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


def get_skills_from_job(job_skills):
    skills = str(job_skills).split(",")

    skills = [clean_text(skill) for skill in skills]
    skills = [skill for skill in skills if skill]

    # Remove duplicate skills
    skills = list(dict.fromkeys(skills))

    return skills


def find_skill_gap(resume_text, job_skills):
    resume_text = clean_text(resume_text)
    required_skills = get_skills_from_job(job_skills)

    matched_skills = []
    missing_skills = []

    for skill in required_skills:
        # Match the complete skill phrase
        pattern = r'\b' + re.escape(skill) + r'\b'

        if re.search(pattern, resume_text):
            matched_skills.append(skill)
        else:
            missing_skills.append(skill)

    return matched_skills, missing_skills

In [13]:
for index, row in recommendations.iterrows():
    matched_skills, missing_skills = find_skill_gap(
        test_resume,
        row["key_skills"]
    )

    print("\nJob:", row["job_role"])
    print("Match Score:", round(row["match_score"], 2), "%")
    print("Matched Skills:", matched_skills)
    print("Missing Skills:", missing_skills)


Job: Chief Financial Officer
Match Score: 26.87 %
Matched Skills: ['management accounting', 'financial management', 'finance', 'management', 'accounting']
Missing Skills: ['accounts and finance', 'account management']

Job: Finance Control Manager
Match Score: 25.48 %
Matched Skills: ['management accounting', 'financial management', 'management', 'control']
Missing Skills: ['accounts reconciliation', 'financial accounting', 'budget management', 'budgetary analysis']

Job: Accounts Associate
Match Score: 25.14 %
Matched Skills: ['accounting operations', 'accounting', 'operations', 'finance']
Missing Skills: ['financial accounting']

Job: Manager Finance and Accounts
Match Score: 24.91 %
Matched Skills: ['general accounting', 'accounting functions', 'financial management', 'general']
Missing Skills: ['team management', 'accounts and finance', 'accounts finalisation', 'financial operations']

Job: Accountant Staff
Match Score: 24.71 %
Matched Skills: ['management accounting', 'general ac

In [14]:
from collections import Counter

all_required_skills = []

for _, row in recommendations.iterrows():
    skills = get_skills_from_job(row["key_skills"])
    all_required_skills.extend(skills)

skill_frequency = Counter(all_required_skills)

resume_cleaned = clean_text(test_resume)

matched_overall = []
missing_overall = []

for skill, count in skill_frequency.most_common():
    pattern = r'\b' + re.escape(skill) + r'\b'

    if re.search(pattern, resume_cleaned):
        matched_overall.append(skill)
    else:
        missing_overall.append(skill)

print("Overall Skill Gap Report")
print("------------------------")

print("\nSkills found in resume:")
print(matched_overall[:15])

print("\nSkills to improve:")
print(missing_overall[:15])

Overall Skill Gap Report
------------------------

Skills found in resume:
['accounting', 'management accounting', 'financial management', 'finance', 'management', 'accounting operations', 'general accounting', 'financial reporting', 'control', 'operations', 'accounting functions', 'general', 'account reconciliations', 'general ledger', 'accounts payable']

Skills to improve:
['financial accounting', 'accounts and finance', 'account management', 'accounts reconciliation', 'accounts finalisation', 'financial operations', 'finance and accounts', 'budget management', 'budgetary analysis', 'team management', 'corporate accounting', 'branch accounting', 'balance sheet finalisation', 'english', 'accounting software']


In [19]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(job_tfidf, "../models/job_tfidf.pkl")
joblib.dump(job_matrix, "../models/job_matrix.pkl")
print("Job TF-IDF matrix saved successfully!")
print("Job recommendation TF-IDF model saved successfully!")

Job TF-IDF matrix saved successfully!
Job recommendation TF-IDF model saved successfully!


In [16]:
os.makedirs("../data/processed", exist_ok=True)
jobs.to_csv("../data/processed/cleaned_jobs.csv", index=False)
print("Cleaned job dataset saved successfully!")

Cleaned job dataset saved successfully!


In [18]:
print("Checking saved files...\n")

files_to_check = [
    "../data/processed/cleaned_resumes.csv",
    "../data/processed/cleaned_jobs.csv",
    "../models/resume_classifier.pkl",
    "../models/resume_tfidf.pkl",
    "../models/job_tfidf.pkl"
]

for file in files_to_check:
    if os.path.exists(file):
        print("✓", file)
    else:
        print("✗ Missing:", file)

Checking saved files...

✓ ../data/processed/cleaned_resumes.csv
✓ ../data/processed/cleaned_jobs.csv
✓ ../models/resume_classifier.pkl
✓ ../models/resume_tfidf.pkl
✓ ../models/job_tfidf.pkl


In [20]:
def calculate_precision_at_k(resume_text, recommended_jobs, k=10):
    resume_text = clean_text(resume_text)

    relevant_jobs = 0

    for _, row in recommended_jobs.head(k).iterrows():

        job_skills = get_skills_from_job(row["key_skills"])

        matched_skills = 0

        for skill in job_skills:
            pattern = r'\b' + re.escape(skill) + r'\b'

            if re.search(pattern, resume_text):
                matched_skills += 1

        # Consider a job relevant if at least one required skill
        # is found in the resume
        if matched_skills > 0:
            relevant_jobs += 1

    precision_at_k = relevant_jobs / k

    return precision_at_k


precision_at_10 = calculate_precision_at_k(
    test_resume,
    recommendations,
    k=10
)
print("Recommender Evaluation")
print("----------------------")
print("Precision@10:", round(precision_at_10 * 100, 2), "%")

Recommender Evaluation
----------------------
Precision@10: 100.0 %
